In [29]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, sum as spark_sum, when, expr, lit
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType

26/03/14 06:21:02 WARN BlockManagerMasterEndpoint: No more replicas available for broadcast_72_piece0 !
26/03/14 06:21:02 WARN BlockManagerMaster: Failed to remove shuffle 29 - org.apache.spark.SparkException: Could not find BlockManagerEndpoint1.
	at org.apache.spark.rpc.netty.Dispatcher.postMessage(Dispatcher.scala:178)
	at org.apache.spark.rpc.netty.Dispatcher.postRemoteMessage(Dispatcher.scala:136)
	at org.apache.spark.rpc.netty.NettyRpcHandler.receive(NettyRpcEnv.scala:683)
	at org.apache.spark.network.server.TransportRequestHandler.processRpcRequest(TransportRequestHandler.java:163)
	at org.apache.spark.network.server.TransportRequestHandler.handle(TransportRequestHandler.java:109)
	at org.apache.spark.network.server.TransportChannelHandler.channelRead0(TransportChannelHandler.java:140)
	at org.apache.spark.network.server.TransportChannelHandler.channelRead0(TransportChannelHandler.java:53)
	at io.netty.channel.SimpleChannelInboundHandler.channelRead(SimpleChannelInboundHandler.j

In [2]:
print(f"Spark version: {spark.version}")
print(f"Master: {spark.sparkContext.master}")
print(f"Available cores: {spark.sparkContext.defaultParallelism}")

Spark version: 3.5.3
Master: yarn
Available cores: 2


In [3]:
BUCKET      = "recosys-data-bucket"
SAMPLE_PATH = f"gs://{BUCKET}/samples/events_500k_users.parquet"

# Load the sample
df = spark.read.parquet(SAMPLE_PATH)
print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")
print(f"Columns : {df.columns}")

Rows    : 502,920
Columns : 9
Columns : ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']


In [4]:
df.printSchema()

root
 |-- event_time: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)



In [5]:
# event_time is being read as string, so converting it to proper timestamp
df = df.withColumn(
    "event_time",
    F.to_timestamp("event_time", "yyyy-MM-dd HH:mm:ss z")
)

df.printSchema()

root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)



In [6]:
df.show(5, truncate=False)

+-------------------+----------+----------+-------------------+------------------+------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code     |brand |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+------------------+------+-------+---------+------------------------------------+
|2019-10-01 00:00:01|view      |1307067   |2053013558920217191|computers.notebook|lenovo|251.74 |550050854|7c90fc70-0e80-4590-96f3-13c02c18c713|
|2019-10-01 00:00:19|view      |1306631   |2053013558920217191|computers.notebook|hp    |580.89 |550050854|7c90fc70-0e80-4590-96f3-13c02c18c713|
|2019-10-01 00:01:05|view      |1306083   |2053013558920217191|computers.notebook|hp    |1512.78|550050854|7c90fc70-0e80-4590-96f3-13c02c18c713|
|2019-10-01 00:01:42|view      |1306631   |2053013558920217191|computers.notebook|hp    |580.89 |550050854|7c90fc70-0e80-4590-96f3

In [7]:
df.groupBy("event_type") \
  .count() \
  .orderBy("count", ascending=False) \
  .show()

+----------+------+
|event_type| count|
+----------+------+
|      view|482709|
|      cart| 11364|
|  purchase|  8847|
+----------+------+



In [8]:
# Check NULL values

df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+
|event_time|event_type|product_id|category_id|category_code|brand|price|user_id|user_session|
+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+
|         0|         0|         0|          0|       159545|74109|    0|      0|           0|
+----------+----------+----------+-----------+-------------+-----+-----+-------+------------+



In [9]:
# Check price distribution 

df.select("price").summary(
    "min", "25%", "50%", "75%", "90%", "95%", "99%", "max", "mean"
).show()

+-------+------------------+
|summary|             price|
+-------+------------------+
|    min|               0.0|
|    25%|             64.35|
|    50%|             159.8|
|    75%|             347.5|
|    90%|            732.18|
|    95%|           1007.97|
|    99%|           1741.34|
|    max|           2574.07|
|   mean|286.15965437842937|
+-------+------------------+



In [10]:
zero_price = df.filter(col("price") <= 0).count()
below_floor = df.filter(col("price") < 1).count()
total = df.count()

print(f"Total rows           : {total:,}")
print(f"Zero/negative price  : {zero_price:,}  ({zero_price/total*100:.2f}%)")
print(f"Below $1.00          : {below_floor:,}  ({below_floor/total*100:.2f}%)")

Total rows           : 502,920
Zero/negative price  : 821  (0.16%)
Below $1.00          : 881  (0.18%)


In [11]:
user_activity = df.groupBy("user_id") \
                  .count() \
                  .withColumnRenamed("count", "n_events")

user_activity.select("n_events").summary(
    "min", "25%", "50%", "75%", "90%", "99%", "max", "mean"
).show()

+-------+-----------------+
|summary|         n_events|
+-------+-----------------+
|    min|                5|
|    25%|                7|
|    50%|               13|
|    75%|               28|
|    90%|               57|
|    99%|              194|
|    max|             1482|
|   mean|26.39584317430326|
+-------+-----------------+



In [12]:
user_activity_buckets = user_activity.withColumn(
    "bucket",
    when(col("n_events") <= 5,    "1-5") \
    .when(col("n_events") <= 10,  "6-10") \
    .when(col("n_events") <= 20,  "11-20") \
    .when(col("n_events") <= 50,  "21-50") \
    .when(col("n_events") <= 100, "51-100") \
    .when(col("n_events") <= 500, "101-500") \
    .otherwise(                   "500+")
).groupBy("bucket") \
 .count() \
 .withColumnRenamed("count", "n_users") \
 .orderBy("bucket")

user_activity_buckets.show()

+-------+-------+
| bucket|n_users|
+-------+-------+
|    1-5|   1926|
|101-500|    705|
|  11-20|   4810|
|  21-50|   4344|
|   500+|     24|
| 51-100|   1515|
|   6-10|   5729|
+-------+-------+



In [13]:
user_activity_buckets.orderBy(
    expr("CAST(split(bucket, '-')[0] AS INT)")
).show()

+-------+-------+
| bucket|n_users|
+-------+-------+
|   500+|     24|
|    1-5|   1926|
|   6-10|   5729|
|  11-20|   4810|
|  21-50|   4344|
| 51-100|   1515|
|101-500|    705|
+-------+-------+



In [14]:
bucket_order = {"1-5": 1, "6-10": 2, "11-20": 3, 
                "21-50": 4, "51-100": 5, "101-500": 6, "500+": 7}

user_activity_buckets.withColumn(
    "order",
    when(col("bucket") == "1-5",    1)
    .when(col("bucket") == "6-10",  2)
    .when(col("bucket") == "11-20", 3)
    .when(col("bucket") == "21-50", 4)
    .when(col("bucket") == "51-100",5)
    .when(col("bucket") == "101-500",6)
    .otherwise(7)
).orderBy("order") \
 .drop("order") \
 .show()

+-------+-------+
| bucket|n_users|
+-------+-------+
|    1-5|   1926|
|   6-10|   5729|
|  11-20|   4810|
|  21-50|   4344|
| 51-100|   1515|
|101-500|    705|
|   500+|     24|
+-------+-------+



In [15]:
product_activity = df.groupBy("product_id") \
                     .count() \
                     .withColumnRenamed("count", "n_events")

product_activity.select("n_events").summary(
    "min", "25%", "50%", "75%", "90%", "99%", "max", "mean"
).show()

+-------+-----------------+
|summary|         n_events|
+-------+-----------------+
|    min|                1|
|    25%|                1|
|    50%|                2|
|    75%|                6|
|    90%|               16|
|    99%|              105|
|    max|             5639|
|   mean|9.429985749643741|
+-------+-----------------+



In [16]:
product_activity_buckets = product_activity.withColumn(
    "bucket",
    when(col("n_events") == 1,      "1") \
    .when(col("n_events") <= 3,     "2-3") \
    .when(col("n_events") <= 5,     "4-5") \
    .when(col("n_events") <= 10,    "6-10") \
    .when(col("n_events") <= 50,    "11-50") \
    .when(col("n_events") <= 100,   "51-100") \
    .otherwise(                     "100+")
).groupBy("bucket") \
 .count() \
 .withColumnRenamed("count", "n_products") \
 .withColumn(
    "order",
    when(col("bucket") == "1",      1)
    .when(col("bucket") == "2-3",   2)
    .when(col("bucket") == "4-5",   3)
    .when(col("bucket") == "6-10",  4)
    .when(col("bucket") == "11-50", 5)
    .when(col("bucket") == "51-100",6)
    .otherwise(7)
).orderBy("order") \
 .drop("order")

product_activity_buckets.show()


+------+----------+
|bucket|n_products|
+------+----------+
|     1|     18185|
|   2-3|     14663|
|   4-5|      6025|
|  6-10|      6354|
| 11-50|      6726|
|51-100|       808|
|  100+|       571|
+------+----------+



In [17]:
session_lengths = df.groupBy("user_session") \
                    .count() \
                    .withColumnRenamed("count", "session_length")

session_lengths.select("session_length").summary(
    "min", "25%", "50%", "75%", "90%", "99%", "max", "mean"
).show()

+-------+------------------+
|summary|    session_length|
+-------+------------------+
|    min|                 1|
|    25%|                 1|
|    50%|                 3|
|    75%|                 6|
|    90%|                12|
|    99%|                35|
|    max|               186|
|   mean|5.4302218863035145|
+-------+------------------+



In [18]:
session_buckets = session_lengths.withColumn(
    "bucket",
    when(col("session_length") == 1,      "1") \
    .when(col("session_length") <= 3,     "2-3") \
    .when(col("session_length") <= 5,     "4-5") \
    .when(col("session_length") <= 10,    "6-10") \
    .when(col("session_length") <= 20,    "11-20") \
    .when(col("session_length") <= 50,    "21-50") \
    .otherwise(                           "50+")
).groupBy("bucket") \
 .count() \
 .withColumnRenamed("count", "n_sessions") \
 .withColumn(
    "order",
    when(col("bucket") == "1",     1)
    .when(col("bucket") == "2-3",  2)
    .when(col("bucket") == "4-5",  3)
    .when(col("bucket") == "6-10", 4)
    .when(col("bucket") == "11-20",5)
    .when(col("bucket") == "21-50",6)
    .otherwise(7)
).orderBy("order") \
 .drop("order")

session_buckets.show()

+------+----------+
|bucket|n_sessions|
+------+----------+
|     1|     25526|
|   2-3|     24935|
|   4-5|     14373|
|  6-10|     15878|
| 11-20|      8418|
| 21-50|      3171|
|   50+|       314|
+------+----------+



In [19]:
hourly = df.withColumn("hour", F.hour("event_time")) \
           .groupBy("hour") \
           .count() \
           .withColumnRenamed("count", "n_events") \
           .orderBy("hour")

hourly.show(24)

+----+--------+
|hour|n_events|
+----+--------+
|   0|    3358|
|   1|    6588|
|   2|   12562|
|   3|   19222|
|   4|   22294|
|   5|   25368|
|   6|   27344|
|   7|   27414|
|   8|   27901|
|   9|   26931|
|  10|   26967|
|  11|   26183|
|  12|   25777|
|  13|   27782|
|  14|   31659|
|  15|   34439|
|  16|   37093|
|  17|   32791|
|  18|   25658|
|  19|   15907|
|  20|    9316|
|  21|    5002|
|  22|    2857|
|  23|    2507|
+----+--------+



In [20]:
daily = df.withColumn(
    "day_of_week",
    F.date_format("event_time", "EEEE")
).groupBy("day_of_week") \
 .count() \
 .withColumnRenamed("count", "n_events") \
 .withColumn(
    "order",
    when(col("day_of_week") == "Monday",    1)
    .when(col("day_of_week") == "Tuesday",  2)
    .when(col("day_of_week") == "Wednesday",3)
    .when(col("day_of_week") == "Thursday", 4)
    .when(col("day_of_week") == "Friday",   5)
    .when(col("day_of_week") == "Saturday", 6)
    .otherwise(7)
).orderBy("order") \
 .drop("order")

daily.show()

+-----------+--------+
|day_of_week|n_events|
+-----------+--------+
|     Monday|   63295|
|    Tuesday|   80945|
|  Wednesday|   78927|
|   Thursday|   76774|
|     Friday|   66022|
|   Saturday|   67141|
|     Sunday|   69816|
+-----------+--------+



# EDA actual

In [23]:
BUCKET      = "recosys-data-bucket"
RAW_PATH    = f"gs://{BUCKET}/raw/*.csv"
OUTPUT_PATH = f"gs://{BUCKET}/processed/events_clean"

schema = StructType([
    StructField("event_time",    StringType(),  True),
    StructField("event_type",    StringType(),  True),
    StructField("product_id",    LongType(),    True),
    StructField("category_id",   LongType(),    True),
    StructField("category_code", StringType(),  True),
    StructField("brand",         StringType(),  True),
    StructField("price",         DoubleType(),  True),
    StructField("user_id",       LongType(),    True),
    StructField("user_session",  StringType(),  True),
])

df_raw = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(RAW_PATH)

df_raw = df_raw.withColumn(
    "event_time",
    F.to_timestamp("event_time", "yyyy-MM-dd HH:mm:ss z")
)

print(f"Partitions : {df_raw.rdd.getNumPartitions()}")
print(f"Columns    : {df_raw.columns}")
df_raw.printSchema()

Partitions : 422
Columns    : ['event_time', 'event_type', 'product_id', 'category_id', 'category_code', 'brand', 'price', 'user_id', 'user_session']
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_session: string (nullable = true)



In [24]:
total_rows = df_raw.count()
print(f"Total rows loaded : {total_rows:,}")
print(f"Expected          : 288,779,227")
print(f"Match             : {total_rows == 288_779_227}")

Total rows loaded : 411,709,736
Expected          : 288,779,227
Match             : False


In [25]:
EDA_FILES = [
    f"gs://{BUCKET}/raw/2019-Oct.csv",
    f"gs://{BUCKET}/raw/2019-Nov.csv",
    f"gs://{BUCKET}/raw/2019-Dec.csv",
    f"gs://{BUCKET}/raw/2020-Jan.csv",
    f"gs://{BUCKET}/raw/2020-Feb.csv",
]

df_raw = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv(EDA_FILES)

df_raw = df_raw.withColumn(
    "event_time",
    F.to_timestamp("event_time", "yyyy-MM-dd HH:mm:ss z")
)

total_rows = df_raw.count()
print(f"Total rows loaded : {total_rows:,}")
print(f"Expected          : 288,779,227")
print(f"Match             : {total_rows == 288_779_227}")

Total rows loaded : 288,779,227
Expected          : 288,779,227
Match             : True


In [27]:
# Filling NULL values 

df_step1 = df_raw \
    .fillna("unknown", subset=["category_code", "brand"]) \
    .filter(F.col("user_session").isNotNull())

rows_step1 = df_step1.count()
removed    = 288_779_227 - rows_step1

print(f"Rows before : 288,779,227")
print(f"Rows after  : {rows_step1:,}")
print(f"Removed     : {removed:,}  (null user_session rows)")
print(f"Expected to remove : 66")

Rows before : 288,779,227
Rows after  : 288,779,161
Removed     : 66  (null user_session rows)
Expected to remove : 66


In [28]:
#Drop exact duplicates 

df_step2 = df_step1.dropDuplicates([
    "event_time",
    "event_type", 
    "product_id",
    "user_id",
    "user_session"
])

rows_step2 = df_step2.count()
removed    = rows_step1 - rows_step2

print(f"Rows before : {rows_step1:,}")
print(f"Rows after  : {rows_step2:,}")
print(f"Removed     : {removed:,}  (exact duplicates)")

Rows before : 288,779,161
Rows after  : 287,694,778
Removed     : 1,084,383  (exact duplicates)


In [30]:
w = Window.partitionBy("user_id", "product_id", "event_type") \
          .orderBy("event_time")

df_step3 = df_step2 \
    .withColumn("prev_time", F.lag("event_time").over(w)) \
    .withColumn("gap_secs",
        F.when(F.col("prev_time").isNotNull(),
               F.col("event_time").cast("long") - 
               F.col("prev_time").cast("long"))
    ) \
    .filter(
        F.col("prev_time").isNull() | 
        (F.col("gap_secs") > 1)
    ) \
    .drop("prev_time", "gap_secs")

rows_step3 = df_step3.count()
removed    = rows_step2 - rows_step3

print(f"Rows before : {rows_step2:,}")
print(f"Rows after  : {rows_step3:,}")
print(f"Removed     : {removed:,}  (near-duplicates within 1s)")

Rows before : 287,694,778
Rows after  : 286,856,207
Removed     : 838,571  (near-duplicates within 1s)


In [31]:
df_step4 = df_step3.filter(F.col("price") >= 1.0)

rows_step4 = df_step4.count()
removed    = rows_step3 - rows_step4

print(f"Rows before : {rows_step3:,}")
print(f"Rows after  : {rows_step4:,}")
print(f"Removed     : {removed:,}  (price below $1.00)")

Rows before : 286,856,207
Rows after  : 286,230,754
Removed     : 625,453  (price below $1.00)


In [ ]:
user_avg_epd = df_step4 \
    .withColumn("day", F.to_date("event_time")) \
    .groupBy("user_id", "day") \
    .agg(F.count("*").alias("daily_events")) \
    .groupBy("user_id") \
    .agg(F.avg("daily_events").alias("avg_daily_events")) \
    .filter(F.col("avg_daily_events") > 300) \
    .select("user_id")

n_bots = user_avg_epd.count()

df_step5 = df_step4.join(user_avg_epd, "user_id", "leftanti")

rows_step5 = df_step5.count()
removed    = rows_step4 - rows_step5

print(f"Bot users found : {n_bots:,}")
print(f"Rows before     : {rows_step4:,}")
print(f"Rows after      : {rows_step5:,}")
print(f"Removed         : {removed:,}  (bot user events)")